In [0]:
# =====================================================================
# proceso / 02_ingest_ecommerce.py
# =====================================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("raw_path", "abfss://raw@adlsretailproject0826.dfs.core.windows.net/ecommerce/")
dbutils.widgets.text("catalogo", "retail_medallion")
raw_path = dbutils.widgets.get("raw_path")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
ecommerce_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Product_Category", StringType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Customer_Segment", StringType(), True),
    StructField("Marketing_Spend", DoubleType(), True),
    StructField("Units_Sold", IntegerType(), True),
])

In [0]:
df_raw = (
    spark.read
    .option("header", True)
    .schema(ecommerce_schema)
    .csv(raw_path)
)

df_bronze = (
    df_raw
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingest_timestamp", F.current_timestamp())
    .withColumn("_source_system", F.lit("ecommerce_kaggle"))
    .select(
        "Date", "Product_Category", "Price", "Discount", "Customer_Segment",
        "Marketing_Spend", "Units_Sold",
        "_source_file", "_ingest_timestamp", "_source_system",
    )
)

In [0]:
df_bronze.write.mode("overwrite").insertInto(f"{catalogo}.bronze.ecommerce_raw")

print(f"Bronze OK -> {catalogo}.bronze.ecommerce_raw ({df_bronze.count()} filas)")